# PyTorch `nn.Embedding`과 사전학습 벡터 초기화

신경망은 문자열 토큰을 직접 계산하지 못하므로 먼저 각 토큰을 정수 ID로 바꾸고, 그 ID를 의미 특성으로 사용할 실수 벡터에 연결해야 한다.

PyTorch의 **`nn.Embedding`** 은 정수 토큰 ID를 입력받아 `(어휘 수, 임베딩 차원)` 행렬의 해당 행을 찾아 밀집 벡터로 반환하는 계층이다.
원-핫 벡터와 선형계층의 곱을 매번 계산하지 않고 같은 결과를 효율적인 행 조회로 얻으며, 분류 손실의 역전파로 단어 벡터도 작업 목적에 맞게 갱신할 수 있다.


## 01. 토큰 ID와 패딩 준비

`nn.Embedding`은 문자열이 아니라 **토큰 ID**를 입력받는다. 토큰 ID는 어휘 사전에서 각 토큰이 대응하는 정수이자 임베딩 행렬에서 조회할 행 번호이다. 어휘에 없는 단어는 **OOV(Out Of Vocabulary)** 특수 토큰의 ID로 바꾸므로, 처음 보는 입력도 정해진 행 범위 안에서 처리할 수 있다.

길이가 다른 여러 문장을 하나의 배치 텐서로 묶으려면 짧은 문장의 빈 위치를 **PAD** 특수 토큰으로 채우는 **패딩(Padding)** 이 필요하다.

In [2]:
import torch

# 어휘 사전 (단어 - ID)
# ID가 임베딩 행렬의 행 번호가 된다.
word_to_id = {
    "<PAD>": 0,
    "<OOV>": 1,
    "movie": 2,
    "story": 3,
    "actor": 4,
    "good": 5,
    "great": 6,
    "fun": 7,
    "bad": 8,
    "boring": 9,
    "awful": 10,
}

tokenized_sentences = [
    ["movie", "good"],
    ["story", "great"],
    ["actor", "good"],
    ["movie", "good", "fun"],
    ["movie", "bad"],
    ["story", "boring"],
    ["actor", "awful"],
    ["movie", "bad", "boring"],
]

# 지도 학습 용 정답
# 1은 긍정, 0은 부정
labels = torch.tensor([1, 1, 1, 1, 0, 0, 0, 0], dtype=torch.long)

# 최대 토큰 수 구하기 -> 패딩 후 공통 문장 길이에서 사용
max_length = max(len(tokens) for tokens in tokenized_sentences)

# 토큰 문자열을 토큰 ID로 변환 + <PAD> 0 추가
encoded_sentences = [
    [word_to_id.get(token, word_to_id["<OOV>"]) for token in tokens]
    + [word_to_id["<PAD>"]] * (max_length - len(tokens))
    for tokens in tokenized_sentences
]

# 2차원 리스트 -> Tensor로 변경
input_ids = torch.tensor(encoded_sentences, dtype=torch.long)

print("입력 ID shape:", input_ids.shape)
print("첫 문장 토큰:", tokenized_sentences[0])
print("첫 문장 ID:", input_ids[0].tolist())
print("정답 shape:", labels.shape)


# 입력 ID shape: [8, 3] == 토큰 ID 목록 8행 3열
# 첫 문장 ID: [2, 5, 0] == ['movie', 'good'], '<PAD>'] id 목록
# 정답 shape: [8] == [1,1,1,1,0,0,0,0] 각 리뷰 긍정/부정 정답 목록

입력 ID shape: torch.Size([8, 3])
첫 문장 토큰: ['movie', 'good']
첫 문장 ID: [2, 5, 0]
정답 shape: torch.Size([8])


## 02. `nn.Embedding`으로 벡터 조회

`nn.Embedding(num_embeddings, embedding_dim)`은 `(어휘 수, 임베딩 차원)`의 학습 가능한 가중치 행렬을 만든다. `num_embeddings`는 사용할 수 있는 토큰 ID의 개수이자 행 수이고, `embedding_dim`은 단어 하나를 표현하는 실수 특성의 수이다. 입력 ID와 같은 번호의 행 벡터를 가져오는 연산을 **임베딩 조회(Embedding Lookup)** 라고 한다.

`padding_idx`에는 PAD 행 번호를 지정하며 이 행은 기본적으로 0 벡터로 유지돼 학습 중 다른 단어처럼 갱신되지 않는다. 입력 shape이 `(8, 3)`이고 임베딩 차원이 4이면 실행 시 예상되는 출력은 `(8, 3, 4)`이다. 문장 수와 토큰 위치 축은 유지되고 마지막에 임베딩 축만 추가되며, `embedding.weight[2]`와 첫 문장 첫 토큰의 출력이 같아야 ID 2가 행 2를 조회했다는 설명이 성립한다.

![토큰 ID가 임베딩 행을 조회하고 임베딩 축이 추가되는 과정](attachment:torch_embedding_lookup.png)

그림에서 `[2, 5, 0]`은 값의 크기가 아니라 `weight`에서 조회할 행 번호이다. 전체 배치의 문장 수와 토큰 위치는 유지되고 임베딩 차원 4가 새 축으로 추가되며, PAD 위치는 사라지지 않고 0벡터로 남아 다음 평균 계산에서 마스크로 제외한다.

In [6]:
from torch import nn

torch.manual_seed(42)

embedding = nn.Embedding(
    # ID가 0부터 지정된 어휘 사전 행 수 만큼의 임베딩 가중치 벡터가 생성된다.
    # 가중치라고 표현하지만 중요도나 빈도를 나타내는 것은 아니고, 각 행(토큰)의 특성을 표기하기 위한 벡터 값이다.
    # 초기 값은 랜덤이고, 학습 시 조금씩 수정된다.
    # 0번 행  → [ 0.0000,  0.0000,  0.0000,  0.0000]
    # 1번 행  → [ 0.6784, -1.2345, -0.0431, -1.6047]
    # 2번 행  → [-0.7521,  1.6487, -0.3925, -1.4036]
    num_embeddings=len(word_to_id), # 토큰의 개수(11개)
    embedding_dim=4, # 토큰 하나를 표현할 차원(feature 수) (4개)
    padding_idx=word_to_id["<PAD>"],
    # PAD 행의 가중치(벡터값) 0으로 유지해서 기울기 막기
)

embedded = embedding(input_ids)

print("임베딩 행렬 shape:", embedding.weight.shape)
print("입력 ID shape:", input_ids.shape)
print("임베딩 출력 shape:", embedded.shape)
print("첫 문장 첫 토큰 ID:", input_ids[0, 0].item())
print("movie 행과 출력 동일:", torch.allclose(embedding.weight[2], embedded[0, 0]))

# 임베딩 표에서 토큰 아이디 2번 값과,
# 토큰 아이디 2번을 임베딩표와 연결한 값이
print(embedding.weight[2]) # 임베딩 표

print(embedded[0, 0]) # input_ids (문장별 토큰 ID + 임베딩 표의 벡터값)

임베딩 행렬 shape: torch.Size([11, 4])
입력 ID shape: torch.Size([8, 3])
임베딩 출력 shape: torch.Size([8, 3, 4])
첫 문장 첫 토큰 ID: 2
movie 행과 출력 동일: True
tensor([-0.7521,  1.6487, -0.3925, -1.4036], grad_fn=<SelectBackward0>)
tensor([-0.7521,  1.6487, -0.3925, -1.4036], grad_fn=<SelectBackward0>)


### PAD 벡터와 문장 평균 마스크

`padding_idx=0`으로 지정한 PAD 행은 0 벡터로 초기화되고 역전파에서도 갱신되지 않는다. 그러나 문장 평균을 단순히 전체 길이로 나누면 PAD가 추가된 문장의 실제 토큰 벡터가 불필요하게 작아질 수 있다.

마스크는 실제 토큰 위치를 1, PAD 위치를 0으로 표시한다. 실제 토큰 벡터의 합을 실제 토큰 수로 나누면 패딩 길이와 관계없이 문장 평균을 계산할 수 있다. 코드를 실행했을 때 PAD 벡터가 길이 4의 0 벡터이고 `(8, 3)` 마스크가 토큰 위치 축을 줄여 `(8, 4)` 문장 벡터를 만들면, PAD가 평균의 분모와 분자에서 모두 제외된 것이다.

In [8]:
token_mask = input_ids.ne(word_to_id["<PAD>"])

expanded_mask = token_mask.unsqueeze(-1)

masked_sum = (embedded * expanded_mask).sum(dim=1)
print(masked_sum)

real_token_count = expanded_mask.sum(dim=1).clamp(min=1)

# 각 문장의 벡터 평균(8,4)
# 단, 평균 계산 시 <PAD>는 제외
sentence_vectors = masked_sum / real_token_count


print("PAD 벡터:", embedding.weight[word_to_id["<PAD>"]].tolist())
print("마스크 shape:", token_mask.shape)
print("문장 벡터 shape:", sentence_vectors.shape)


PAD 벡터: [0.0, 0.0, 0.0, 0.0]
마스크 shape: torch.Size([8, 3])
문장 벡터 shape: torch.Size([8, 4])


## 03. 평균 임베딩 문장 분류기

분류 모델은 임베딩 조회, PAD를 제외한 평균, 선형 분류를 하나의 `nn.Module`로 묶는다. `forward()`는 입력 텐서가 이 계층들을 통과하는 순서를 정의한다. 여러 토큰 벡터를 평균해 문장 벡터 하나로 줄이는 처리를 **평균 풀링(Mean Pooling)** 이라고 하며, 계산이 빠르지만 토큰 순서는 보존하지 않는다.

`nn.Linear`는 문장 평균 벡터를 클래스별 **로짓(Logit)** 으로 바꾼다. 로짓은 소프트맥스를 적용하기 전의 원시 점수이며, 이번 모델은 부정과 긍정 두 클래스에 대해 문장마다 두 값을 반환한다. 코드를 실행해 초기 로짓 shape이 `(8, 2)`라면 첫 번째 축은 문장 8개, 두 번째 축은 클래스 2개라는 입출력 계약이 맞는 것이다.

In [15]:
class MeanEmbeddingClassifier(nn.Module):
    def __init__(self, vocabulary_size, embedding_dim, padding_id, class_count):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocabulary_size, # 11개
            embedding_dim=embedding_dim, # 4
            padding_idx=padding_id, # <PAD> == 0
        )

        # 평균으로 만든 문장 벡터를 class_count 개의 logit으로 변경하는 계층
        # 입력 : 각 문장의 토큰 3개의 벡터 평균값
        #       단, pad는 제외
        self.classifier = nn.Linear(embedding_dim, class_count)

        # 같은 객체 내에서 사용할 인스턴스 변수 선언
        self.padding_id = padding_id


    # 순전파
    # batch_input_ids (배치크기, 토큰 위치 수)
    def forward(self, batch_input_ids):

        # 각 ID가 벡터로 변경됨(배치크기, 토큰 위치 수, 임베딩 차원)
        #  == (문장 수, 토늨 수, 벡터 수) == (8, 3, 4)
        embedded = self.embedding(batch_input_ids)


        # PAD(0)이 아닌 위치를 True로 표시하고
        # 차원 크기를 맞추기 위해서 마지막 차원 1 추가
        # == (8, 3) -> (8 ,3, 1)
        mask = batch_input_ids.ne(self.padding_id).unsqueeze(-1)


        # (8 ,3, 4) * (8 ,3, 1) -> 브로드캐스팅 -> (8, 3, 4) * (8, 3, 4)
        # sum(dim =1) -> 행(토큰 수 == 3) 방향으로 합계 구하기
        summed = (embedded * mask).sum(dim=1)

        # 각 문장에서 <PAD> (0)이 아닌 토큰의 개수를 구하기
        token_counts = mask.sum(dim=1).clamp(min=1)

        # 각 문장별 토큰의 합을 <PAD>이 아닌 토큰의 개수로 나누어 평균 계산
        # == [2,5,0]에서 0인 부분을 제외하고 같은 인덱스 벡터 평균 구하기
        # == 평균 문장 벡터
        sentence_vectors = summed / token_counts

        return self.classifier(sentence_vectors)

torch.manual_seed(42)

# 임베딩 후 분류하는 모델 생성
classifier = MeanEmbeddingClassifier(
    vocabulary_size=len(word_to_id),
    embedding_dim=8,
    padding_id=word_to_id["<PAD>"],
    class_count=2,
)
initial_logits = classifier(input_ids)

print("초기 로짓 shape:", initial_logits.shape)


초기 로짓 shape: torch.Size([8, 2])


### 04. 임베딩과 분류기 함께 학습하기

**손실함수(Loss Function)** 는 현재 예측과 정답의 차이를 하나의 값으로 계산한다. `CrossEntropyLoss`는 `(문장 수, 클래스 수)` 로짓과 `(문장 수,)` 정답 클래스 ID를 비교하며 내부에서 소프트맥스에 해당하는 계산을 수행하므로 모델 출력에 소프트맥스를 먼저 적용하지 않는다. `optim.Adam`은 이 손실의 기울기를 사용해 임베딩 행렬과 선형 분류기 파라미터를 함께 갱신한다.

학습 반복은 `zero_grad()`로 이전 기울기를 지우고, 모델의 로짓과 손실을 계산한 뒤, `backward()`로 기울기를 구하고 `step()`으로 파라미터를 갱신하는 순서이다. 실행 후 출력되는 손실이 전반적으로 감소하면 여덟 학습 문장을 구분하는 방향으로 두 계층이 함께 바뀐 것이다. 반대로 NaN이 나오거나 계속 증가하면 학습률, 입력 ID 범위와 정답 shape을 먼저 점검한다.

이 작은 데이터는 같은 문장을 학습과 평가에 모두 사용하므로 정확도는 코드 흐름 확인용이다. 높은 학습 문장 정확도만으로 새로운 문장에 대한 일반화 성능을 판단할 수 없다.

In [16]:
from torch import optim

# 손실함수
criterion = nn.CrossEntropyLoss() # 내부에 SoftMax() 존재

# 최적화 객체
optimizer = optim.Adam(classifier.parameters(), lr=0.05)

# 학습 모드로 전환
classifier.train()

for epoch in range(1,101): # 100회 반복 학습
    optimizer.zero_grad() # 이전 반복 기울기 초기화

    # input_ids : 각 문장별 토큰 ID 목록 (8,3)
    # logits : 각 문장별 긍정, 부정 점수 (8,2)
    logits = classifier(input_ids)

    # CrossEntropyLoss 내부에 있는 SoftMax에게 logits를 전달
    # -> 활성화된 결과(긍정 확률, 부정 확률) 반환
    # -> 활성화된 결과와 labels(정답)을 비교
    # -> loss 계산
    loss = criterion(logits, labels)

    # 역전파(기울기 계산)
    loss.backward()

    # 파라미터별 가중치 최적화
    optimizer.step()

    if epoch == 1 or epoch % 20 == 0:
        print(f'epoch:{epoch:3d}, loss:{loss.item(): .4f}')

epoch:  1, loss: 0.7950
epoch: 20, loss: 0.0025
epoch: 40, loss: 0.0002
epoch: 60, loss: 0.0001
epoch: 80, loss: 0.0001
epoch:100, loss: 0.0001


### 학습 문장의 예측 확인

평가에서는 기울기 기록을 중단하고 가장 높은 로짓의 클래스 번호를 선택한다. `torch.no_grad()`는 평가 계산 그래프를 만들지 않아 메모리 사용을 줄이며, `argmax(dim=1)`은 클래스 축에서 가장 큰 점수의 인덱스를 반환한다.

여기서는 학습에 사용한 문장을 다시 평가하므로 학습 코드가 동작하는지만 확인한다. 실행 후 정답과 예측 목록의 같은 위치가 얼마나 일치하는지 세어 출력된 정확도를 읽는다. 값이 높더라도 이미 본 여덟 문장을 다시 평가한 결과이므로, 새로운 데이터에 대한 성능은 학습·검증·테스트를 분리해 측정해야 한다.

In [17]:
# 평가 모드로 전환
classifier.eval()

with torch.no_grad(): # 역전파에 사용되는 계산식, 기울기 기록 X -> 메모리 아낌
    logits = classifier(input_ids)

    # 각 문장에서 0 또는 1중 가장 큰 로짓값의 인덱스 반환
    prediction = logits.argmax(dim=1)

print("정답: ", labels.tolist())
print("예측: ", prediction.tolist())

정답:  [1, 1, 1, 1, 0, 0, 0, 0]
예측:  [1, 1, 1, 1, 0, 0, 0, 0]


## 05. 사전학습 벡터로 임베딩 초기화

외부 Word2Vec·GloVe·FastText 벡터를 사용할 때는 현재 `word_to_id`의 행 순서에 맞춰 `(어휘 수, 사전학습 차원)` 행렬을 만든다. 등록 단어의 행에는 사전학습 벡터를 복사하고 PAD·OOV·미등록 단어의 초기값은 작업 규칙에 따라 정한다. 행 순서가 어긋나면 shape은 정상이어도 `movie` ID가 다른 단어 벡터를 조회하는 조용한 오류가 생긴다.

`nn.Embedding.from_pretrained()`는 이렇게 준비한 2차원 행렬로 임베딩 계층을 만든다. `freeze=True`는 사전학습 벡터를 고정해 작은 데이터에서 원래 표현이 무너지는 위험을 줄이고, `freeze=False`는 그 벡터를 시작값으로 사용해 현재 작업에 맞게 미세 조정한다. 데이터가 작고 사전학습 도메인이 잘 맞으면 고정을 먼저 고려하고, 작업 데이터가 충분하거나 도메인 차이가 크면 미세 조정을 검토한다.

아래에서는 파일 다운로드 없이 초기화 원리를 확인하기 위해 모양이 명확한 예제 행렬을 사용한다. 실행 시 두 계층의 shape이 `(11, 4)`이고 `requires_grad`가 각각 `False`, `True`이며 `movie`가 예제 행렬의 2번 행을 가리키면 행 정렬과 고정 여부가 의도대로 설정된 것이다. 실제 작업에서는 같은 자리에 사전학습 모델에서 단어별로 조회한 벡터를 배치한다.

In [20]:
# 임의로 만든 사전학습 벡터(실제 작업에서는 별도 모델 다운로드 또는 공유 받아서 사용)
pretrained_matrix = torch.arange(
    len(word_to_id) * 4,
    dtype=torch.float32,
).reshape(len(word_to_id), 4) / 10

# padding_idx는 기존 행 값을 자동으로 0으로 바꾸지 않으므로 PAD ID에 해당하는 행을 직접 0으로 지정한다.
pretrained_matrix[word_to_id["<PAD>"]] = 0

# from_pretrained()는 준비한 2차원 행렬을 초기 weight로 사용하는 nn.Embedding을 만든다.
frozen_embedding = nn.Embedding.from_pretrained(
    pretrained_matrix,
    # freeze=True는 weight.requires_grad를 False로 만들어 역전파에서 사전학습 벡터를 고정한다.
    freeze=True,
    # PAD 행 번호를 등록해 이 계층을 학습하더라도 해당 행이 갱신되지 않게 한다.
    padding_idx=word_to_id["<PAD>"],
)

# clone()은 고정 계층과 미세 조정 계층이 같은 저장 공간을 공유하지 않도록 독립 행렬을 만든다.
trainable_embedding = nn.Embedding.from_pretrained(
    pretrained_matrix.clone(),
    # freeze=False는 weight.requires_grad를 True로 두어 현재 작업의 손실로 벡터를 미세 조정한다.
    freeze=False,
    padding_idx=word_to_id["<PAD>"],
)

# 두 계층의 행렬 shape과 기울기 갱신 여부를 비교한다.
print("초기화 shape:", frozen_embedding.weight.shape)
print("고정 requires_grad:", frozen_embedding.weight.requires_grad)
print("미세 조정 requires_grad:", trainable_embedding.weight.requires_grad)

# movie의 ID가 2이므로 2번 행이 [0.8, 0.9, 1.0, 1.1]인지 확인해 어휘와 행 정렬을 검증한다.
print("movie 벡터:", frozen_embedding.weight[word_to_id["movie"]].tolist())

초기화 shape: torch.Size([11, 4])
고정 requires_grad: False
미세 조정 requires_grad: True
movie 벡터: [0.800000011920929, 0.8999999761581421, 1.0, 1.100000023841858]


## 06. 선택 기준과 한계

임베딩을 처음부터 학습하면 현재 작업에 맞는 표현을 만들 수 있지만 충분한 데이터가 필요하다. 사전학습 벡터는 작은 데이터에서도 유용한 시작점을 제공하지만 언어·도메인·토큰화 방식이 현재 어휘와 맞아야 한다.

평균 임베딩 모델은 빠르고 해석하기 쉬운 기준선이지만 단어 순서를 잃는다. `good movie`와 `movie good`의 평균은 같으며 부정 범위나 긴 문맥을 충분히 반영하지 못한다. 따라서 이 단원에서 확인한 ID 조회와 PAD 마스킹은 유지하되, 순서가 중요한 작업에서는 RNN·LSTM·Transformer처럼 토큰 위치의 관계를 처리하는 다음 모델로 확장한다.